In [2]:
# https://www.matecdev.com/posts/landsat-sentinel-aws-s3-python.html
from pystac_client import Client
from json import load
import requests
from pyproj import Transformer
import rasterio as rio
import matplotlib.pyplot as plt
import satsearch

In [3]:
LandsatSTAC = Client.open("https://landsatlook.usgs.gov/stac-server", headers=[])

for collection in LandsatSTAC.get_collections():
    print(collection)

<CollectionClient id=landsat-c2l2-sr>
<CollectionClient id=landsat-c2l2-st>
<CollectionClient id=landsat-c2ard-st>
<CollectionClient id=landsat-c2l2alb-bt>
<CollectionClient id=landsat-c2l3-fsca>
<CollectionClient id=landsat-c2ard-bt>
<CollectionClient id=landsat-c2l1>
<CollectionClient id=landsat-c2l3-ba>
<CollectionClient id=landsat-c2l2alb-st>
<CollectionClient id=landsat-c2ard-sr>
<CollectionClient id=landsat-c2l2alb-sr>
<CollectionClient id=landsat-c2l2alb-ta>
<CollectionClient id=landsat-c2l3-dswe>
<CollectionClient id=landsat-c2ard-ta>


In [4]:
def BuildSquare(lon, lat, delta):
    c1 = [lon + delta, lat + delta]
    c2 = [lon + delta, lat - delta]
    c3 = [lon - delta, lat - delta]
    c4 = [lon - delta, lat + delta]
    geometry = {"type": "Polygon", "coordinates": [[ c1, c2, c3, c4, c1 ]]}
    return geometry

geometry = BuildSquare(-59.346271, -34.233076, 0.04)
timeRange = '2019-06-01/2021-06-01'

In [5]:
LandsatSearch = LandsatSTAC.search ( 
    intersects = geometry,
    datetime = timeRange,
    query =  ['eo:cloud_cover95'],
    collections = ["landsat-c2l2-sr"] )

Landsat_items = [i.to_dict() for i in LandsatSearch.items()]
print(f"{len(Landsat_items)} Landsat scenes fetched")

193 Landsat scenes fetched


In [6]:
SentinelSearch = satsearch.Search.search( 
    url = "https://earth-search.aws.element84.com/v0",
    intersects = geometry,
    datetime = timeRange,
    collections = ['sentinel-s2-l2a-cogs'] )

Sentinel_items = SentinelSearch.items()
print(Sentinel_items.summary())

for item in Sentinel_items:
    red_s3 = item.assets['B04']['href']
    print(red_s3)

Items (143):
date                      id                        
2021-05-28                S2B_21HTC_20210528_0_L2A  
2021-05-23                S2A_21HTC_20210523_0_L2A  
2021-05-18                S2B_21HTC_20210518_0_L2A  
2021-05-13                S2A_21HTC_20210513_0_L2A  
2021-05-08                S2B_21HTC_20210508_0_L2A  
2021-05-03                S2A_21HTC_20210503_0_L2A  
2021-04-28                S2B_21HTC_20210428_0_L2A  
2021-04-23                S2A_21HTC_20210423_0_L2A  
2021-04-18                S2B_21HTC_20210418_0_L2A  
2021-04-13                S2A_21HTC_20210413_0_L2A  
2021-04-08                S2B_21HTC_20210408_0_L2A  
2021-04-03                S2A_21HTC_20210403_0_L2A  
2021-03-29                S2B_21HTC_20210329_0_L2A  
2021-03-24                S2A_21HTC_20210324_0_L2A  
2021-03-19                S2B_21HTC_20210319_0_L2A  
2021-03-14                S2A_21HTC_20210314_0_L2A  
2021-03-09                S2B_21HTC_20210309_0_L2A  
2021-03-04                S2A_21H

In [17]:
for item in Landsat_items:
    red_href = item['assets']['red']['href']
    red_s3 = item['assets']['red']['alternate']['s3']['href']
    print(red_href)    
    print(red_s3)
    print(item['assets']['nir08'])

https://landsatlook.usgs.gov/data/collection02/level-2/standard/oli-tirs/2021/225/084/LC08_L2SP_225084_20210528_20210607_02_T1/LC08_L2SP_225084_20210528_20210607_02_T1_SR_B4.TIF
s3://usgs-landsat/collection02/level-2/standard/oli-tirs/2021/225/084/LC08_L2SP_225084_20210528_20210607_02_T1/LC08_L2SP_225084_20210528_20210607_02_T1_SR_B4.TIF
{'href': 'https://landsatlook.usgs.gov/data/collection02/level-2/standard/oli-tirs/2021/225/084/LC08_L2SP_225084_20210528_20210607_02_T1/LC08_L2SP_225084_20210528_20210607_02_T1_SR_B5.TIF', 'type': 'image/vnd.stac.geotiff; cloud-optimized=true', 'title': 'Near Infrared Band 0.8 (B5)', 'description': 'Collection 2 Level-2 Near Infrared Band 0.8 (B5) Surface Reflectance', 'eo:bands': [{'name': 'B5', 'common_name': 'nir08', 'gsd': 30, 'center_wavelength': 0.86}], 'alternate': {'s3': {'storage:platform': 'AWS', 'storage:requester_pays': True, 'href': 's3://usgs-landsat/collection02/level-2/standard/oli-tirs/2021/225/084/LC08_L2SP_225084_20210528_20210607_0

In [8]:
def download_landsat(landsat_url, download_path):
    response = requests.get(landsat_url, stream=True)
    if response.status_code == 200:
        with open(download_path, 'wb') as f:
            for chunk in response.iter_content(1024):
                f.write(chunk)
    else:
        raise Exception(f"Failed to download Landsat data. Status code: {response.status_code}")

In [9]:
from pyproj import Transformer

def getSubset(geotiff_file, bbox):
    with rio.open(geotiff_file) as geo_fp:
        # Calculate pixels with PyProj
        Transf = Transformer.from_crs("epsg:4326", geo_fp.crs)
        lat_north, lon_west = Transf.transform(bbox[3], bbox[0])
        lat_south, lon_east = Transf.transform(bbox[1], bbox[2])
        x_top, y_top = geo_fp.index(lat_north, lon_west)
        x_bottom, y_bottom = geo_fp.index(lat_south, lon_east)
        
        # Define window in RasterIO
        window = rio.windows.Window.from_slices((x_top, x_bottom), (y_top, y_bottom))
        
        # Read the subset
        subset = geo_fp.read(1, window=window)
    
    return subset


In [10]:
def plotNDVI(nir,red,filename):
    ndvi = (nir-red)/(nir+red)
    ndvi[ndvi>1] = 1
    plt.imshow(ndvi)
    plt.savefig(filename)
    plt.close()

In [32]:
from rasterio.features import bounds
import matplotlib.pyplot as plt
import os

bbox = bounds(geometry)


for i,item in enumerate(Landsat_items):
    red_href = item['assets']['red']['href']
    nir_href =  item['assets']['nir08']['href']

    red_path = os.path.join("./", f"{date}_red.tif")
    nir_path = os.path.join("./", f"{date}_nir.tif")
    download_landsat(red_href, red_path)
    download_landsat(nir_href, nir_path)
    date = item['properties']['datetime'][0:10]
    print("Landsat item number " + str(i) + "/" + str(len(Landsat_items)) + " " + date)
    red = getSubset(red_path, bbox)
    nir = getSubset(nir_path, bbox)
    plotNDVI(nir,red,"landsat/" + date + "_ndvi.png")


Landsat item number 0/193 2021-05-28


RasterioIOError: './2021-05-28_red.tif' not recognized as being in a supported file format.